<a href="https://colab.research.google.com/github/MicheleQGF/lsstcorp_DM-TAWOS_test/blob/main/TAWOS_Project_Test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
from pathlib import Path

# Este comando encuentra tu carpeta de usuario real, se llame como se llame
user_home = Path.home()
print("Tu carpeta de usuario real es:", user_home)

Tu carpeta de usuario real es: C:\Users\PC


In [ ]:
# Cambia la ruta por la carpeta que tú quieras
nueva_ruta = r"C:\Users\PC\Documents\TAWOS"

os.chdir(nueva_ruta)

# Verificamos que se haya cambiado
print("Ahora estás en:", os.getcwd())

Ahora estás en: C:\Users\PC\Documents\TAWOS


In [ ]:
import pandas as pd
import numpy as np
# Cargamos el archivo ignorando la advertencia de memoria
df = pd.read_csv('tawos_test_clean.csv', low_memory=False)
df.head()


,issue_id,project_key,project_name,issue_key,type,priority,status,resolution,creation_date,estimation_date,resolution_date,story_point,resolution_time_minutes,in_progress_minutes,total_effort_minutes,creator_id,reporter_id,assignee_id,sprint_id,text
0,14488,MESOS,Apache Mesos,MESOS-10192,Bug,Major,Resolved,Fixed,2020-10-04 04:49:47,NaN,2020-10-13 03:33:25,NaN,12883,0,1,3361.0,3361.0,3360.0,NaN,"""Recent Nvidia CUDA changes break Mesos GPU su..."
1,14491,MESOS,Apache Mesos,MESOS-10189,Task,Critical,Resolved,Fixed,2020-09-15 18:35:51,NaN,2020-09-25 17:46:12,NaN,14350,14349,14349,3359.0,3359.0,3359.0,NaN,"""Pass offer constraints through the V0 schedul..."
2,14497,MESOS,Apache Mesos,MESOS-10183,Bug,Major,Resolved,Invalid,2020-09-02 21:17:16,NaN,2020-09-24 19:54:53,NaN,31597,0,0,3368.0,3368.0,NaN,NaN,"""mesos-master logs incorrectly to error level""..."
3,14501,MESOS,Apache Mesos,MESOS-10179,Task,Major,Resolved,Done,2020-08-20 08:25:20,NaN,2020-09-11 19:16:38,NaN,32331,15744,15744,3359.0,3359.0,3359.0,NaN,"""Expose framework's OfferConstraints via maste..."
4,14503,MESOS,Apache Mesos,MESOS-10177,Task,Major,Resolved,Done,2020-08-17 21:53:43,NaN,2020-09-22 15:24:23,NaN,51450,19985,20427,3359.0,3359.0,3359.0,NaN,"""Add an endpoint for offer constraints debug""\..."


In [ ]:
# Convertimos las columnas de fechas a un formato de fecha real (Datetime)
# 'coerce' transforma los formatos extraños o rotos en Nulls (NaT) para que no rompan el análisis
df['resolution_date'] = pd.to_datetime(df['resolution_date'], errors='coerce')
df['estimation_date'] = pd.to_datetime(df['estimation_date'], errors='coerce')
df['creation_date'] = pd.to_datetime(df['creation_date'], errors='coerce')
df_eval= df.copy()

# escojemos el proyecto que vamos a tratar según su consitencia
# Creamos columnas auxiliares booleanas (True si el dato es válido/real)
df_eval['tiene_creation_date'] = df_eval['creation_date'].notnull()
df_eval['tiene_estimation_date'] = df_eval['estimation_date'].notnull()
df_eval['tiene_resolution_date'] = df_eval['resolution_date'].notnull()
df_eval['tiene_sp_real'] = df_eval['story_point'].notnull()
df_eval['tiene_effort_real'] = (df_eval['total_effort_minutes'] > 0) & (df_eval['total_effort_minutes'].notnull())

# Agrupamos por proyecto y calculamos los porcentajes de éxito
reporte_proyectos = df_eval.groupby('project_name').agg(
    tickets_totales=('issue_key', 'count'),
    pct_fechas_creation_reales=('tiene_creation_date', 'mean'),
    pct_fechas_resolution_reales=('tiene_resolution_date', 'mean'),
    pct_fechas_estimation_reales=('tiene_estimation_date', 'mean'),
    pct_sp_reales=('tiene_sp_real', 'mean'),
    pct_esfuerzo_real=('tiene_effort_real', 'mean')
)

# Convertimos a porcentajes legibles (0-100%)
for col in ['pct_fechas_creation_reales','pct_fechas_resolution_reales', 'pct_fechas_estimation_reales', 'pct_sp_reales', 'pct_esfuerzo_real']:
    reporte_proyectos[col] = (reporte_proyectos[col] * 100).round(1)

# Ordenamos por los que tengan más Story Points reales y volumen
reporte_proyectos = reporte_proyectos.sort_values(by=['pct_sp_reales', 'tickets_totales'], ascending=False)

print("--- EVALUACIÓN DE CALIDAD POR PROYECTO ---")
reporte_proyectos


--- EVALUACIÓN DE CALIDAD POR PROYECTO ---


,tickets_totales,pct_fechas_creation_reales,pct_fechas_resolution_reales,pct_fechas_estimation_reales,pct_sp_reales,pct_esfuerzo_real
project_name,,,,,,
Lsstcorp Data management,10631,100.0,100.0,83.2,83.2,77.9
The MongoDB Engineering,5366,100.0,100.0,50.9,50.9,51.8
DotNetNuke Platform,4047,100.0,100.0,40.0,40.0,52.9
Apache Mesos,3458,100.0,100.0,37.5,37.5,48.0
The Titanium SDK,8740,100.0,100.0,26.8,26.8,61.2
Aptana Studio,1599,100.0,100.0,23.7,23.7,24.8
Atlassian Bamboo,4840,100.0,100.0,5.3,5.3,34.0
Hyperledger Fabric,7159,100.0,100.0,4.1,4.1,68.8
Moodle,28518,100.0,100.0,2.6,2.6,49.4


In [ ]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 133434 entries, 0 to 133433
Data columns (total 20 columns):
 #   Column                   Non-Null Count   Dtype         
---  ------                   --------------   -----         
 0   issue_id                 133434 non-null  int64         
 1   project_key              133434 non-null  str           
 2   project_name             133434 non-null  str           
 3   issue_key                133434 non-null  str           
 4   type                     133434 non-null  str           
 5   priority                 111995 non-null  str           
 6   status                   133434 non-null  str           
 7   resolution               133434 non-null  str           
 8   creation_date            133434 non-null  datetime64[us]
 9   estimation_date          19307 non-null   datetime64[us]
 10  resolution_date          133434 non-null  datetime64[us]
 11  story_point              19307 non-null   float64       
 12  resolution_time_minutes  13

In [ ]:
# Filtramos para dejar únicamente el proyecto más consistente
# Usamos .copy() para que Pandas sepa que es un DataFrame nuevo e independiente
df_lsst = df[df['project_name'] == 'Lsstcorp Data management'].copy()

# Transformaciones clave para proteger la calidad del dato:

# a) Convertimos el esfuerzo de minutos a horas para mejorar la lectura en Tableau
df_lsst['total_effort_hours'] = (df_lsst['total_effort_minutes'] / 60).round(2)

# b) Rellenamos los Story Points vacíos usando la mediana EXCLUSIVA de este proyecto
mediana_lsst_sp = df_lsst['story_point'].median()
df_lsst['story_point'] = df_lsst['story_point'].fillna(mediana_lsst_sp)

# c) Verificamos las dimensiones y salud del nuevo dataset recortado
print(f"¡Filtro aplicado con éxito!")
print(f"Total de tickets para LSST: {df_lsst.shape[0]} filas y {df_lsst.shape[1]} columnas.")
print(f"Mediana de Story Points aplicada a los nulos: {mediana_lsst_sp} puntos.")
print("\nPrimeras filas del proyecto seleccionado:")
df_lsst.head()

¡Filtro aplicado con éxito!
Total de tickets para LSST: 10631 filas y 21 columnas.
Mediana de Story Points aplicada a los nulos: 2.0 puntos.

Primeras filas del proyecto seleccionado:


,issue_id,project_key,project_name,issue_key,type,priority,status,resolution,creation_date,estimation_date,...,story_point,resolution_time_minutes,in_progress_minutes,total_effort_minutes,creator_id,reporter_id,assignee_id,sprint_id,text,total_effort_hours
46840,221116,DM,Lsstcorp Data management,DM-27270,Bug,NaN,Done,Done,2020-10-22 15:16:24,NaT,...,2.0,72,0,47,145715.0,145715.0,145716.0,NaN,"""ap_verify failed on w_2020_43""\n\n""""""A datase...",0.78
46841,221117,DM,Lsstcorp Data management,DM-27269,Story,NaN,Done,Done,2020-10-22 14:51:07,2020-10-22 14:51:07,...,0.2,142,0,62,145717.0,145717.0,145717.0,NaN,"""Move controllers into Controllers directory""\...",1.03
46842,221121,DM,Lsstcorp Data management,DM-27265,Bug,NaN,Invalid,Done,2020-10-21 23:49:02,NaT,...,2.0,25,0,0,145722.0,145722.0,145718.0,NaN,"""No TestID Control on the Connect Vi""\n\n""""""Th...",0.00
46843,221129,DM,Lsstcorp Data management,DM-27256,Story,NaN,Done,Done,2020-10-21 17:42:38,2020-10-21 17:42:38,...,2.0,426,0,206,145720.0,145720.0,145720.0,NaN,"""Add JSON support to butler Config""\n\n""""""Addi...",3.43
46844,221131,DM,Lsstcorp Data management,DM-27254,Story,NaN,Done,Done,2020-10-21 16:41:18,NaT,...,2.0,189,0,0,145732.0,145732.0,145731.0,NaN,"""Get access to NSCA postgres RC2 gen3 database...",0.00


In [ ]:
df_lsst.info()

<class 'pandas.DataFrame'>
RangeIndex: 10631 entries, 46840 to 57470
Data columns (total 21 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   issue_id                 10631 non-null  int64         
 1   project_key              10631 non-null  str           
 2   project_name             10631 non-null  str           
 3   issue_key                10631 non-null  str           
 4   type                     10631 non-null  str           
 5   priority                 0 non-null      str           
 6   status                   10631 non-null  str           
 7   resolution               10631 non-null  str           
 8   creation_date            10631 non-null  datetime64[us]
 9   estimation_date          8849 non-null   datetime64[us]
 10  resolution_date          10631 non-null  datetime64[us]
 11  story_point              10631 non-null  float64       
 12  resolution_time_minutes  10631 non-null

In [ ]:
# Tarea 1: Eliminar la columna priority por estar vacía
df_lsst = df_lsst.drop(columns=['priority'])

# Tarea 2: Rellenar sprint_id faltantes con "No Asignado"
df_lsst['sprint_id'] = df_lsst['sprint_id'].fillna('Not Assigned')

# Tarea 3: Rellenar las fechas de estimación vacías
# Primero aseguramos que ambas columnas se manejen como texto/fechas limpias
df_lsst['estimation_date'] = df_lsst['estimation_date'].astype(str)
df_lsst['resolution_date'] = df_lsst['resolution_date'].astype(str)

# Si 'estimation_date' es nula, vacía o 'nan', tomamos el valor de 'resolution_date'
df_lsst['estimation_date'] = df_lsst.apply(
    lambda row: row['resolution_date'] if pd.isna(row['estimation_date']) or row['estimation_date'] in ['nan', 'None', ''] else row['estimation_date'],
    axis=1
)

# Convertimos ahora sí a formato datetime formal de Pandas
df_lsst['estimation_date'] = pd.to_datetime(df_lsst['estimation_date'], errors='coerce')
df_lsst['resolution_date'] = pd.to_datetime(df_lsst['resolution_date'], errors='coerce')

# --- VERIFICACIÓN ---
print("--- CONTROL DE CALIDAD POST-LIMPIEZA ---")
print(f"¿Quedan nulos en estimation_date?: {df_lsst['estimation_date'].isnull().sum()}")
print(f"¿Quedan nulos en sprint_id?: {df_lsst['sprint_id'].isnull().sum()}")
print(f"\nTotal de registros listos: {df_lsst.shape[0]}")

--- CONTROL DE CALIDAD POST-LIMPIEZA ---
¿Quedan nulos en estimation_date?: 0
¿Quedan nulos en sprint_id?: 0

Total de registros listos: 10631


In [ ]:
df_lsst.info()

<class 'pandas.DataFrame'>
RangeIndex: 10631 entries, 46840 to 57470
Data columns (total 20 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   issue_id                 10631 non-null  int64         
 1   project_key              10631 non-null  str           
 2   project_name             10631 non-null  str           
 3   issue_key                10631 non-null  str           
 4   type                     10631 non-null  str           
 5   status                   10631 non-null  str           
 6   resolution               10631 non-null  str           
 7   creation_date            10631 non-null  datetime64[us]
 8   estimation_date          10631 non-null  datetime64[us]
 9   resolution_date          10631 non-null  datetime64[us]
 10  story_point              10631 non-null  float64       
 11  resolution_time_minutes  10631 non-null  int64         
 12  in_progress_minutes      10631 non-null

In [ ]:
df_lsst.describe()

,issue_id,creation_date,estimation_date,resolution_date,story_point,resolution_time_minutes,in_progress_minutes,total_effort_minutes,creator_id,reporter_id,assignee_id,total_effort_hours
count,10631.000000,10631,10631,10631,10631.000000,10631.000000,10631.000000,10631.000000,10631.000000,10631.000000,10415.000000,10631.000000
mean,233610.451698,2018-03-12 23:38:10.319349,2018-03-15 06:08:57.061518,2018-03-28 02:29:55.329037,3.014065,21771.305898,6391.912990,10603.673972,145782.066692,145778.803311,145778.381661,176.727896
min,221116.000000,2014-03-07 15:25:10,2014-03-07 15:27:33,2014-03-11 11:47:12,0.000000,1.000000,0.000000,0.000000,145714.000000,145714.000000,145714.000000,0.000000
25%,226957.500000,2016-11-04 09:33:30.500000,2016-11-07 11:49:49,2016-11-27 19:41:33,1.000000,1420.500000,0.000000,12.000000,145735.000000,145735.000000,145734.000000,0.200000
50%,233564.000000,2018-05-30 08:08:50,2018-05-30 08:15:41,2018-06-12 13:06:22,2.000000,10422.000000,30.000000,1886.000000,145760.000000,145761.000000,145759.000000,31.430000
75%,239875.500000,2019-09-13 17:39:42.500000,2019-09-16 23:46:18,2019-10-01 20:29:22,4.000000,34503.000000,4908.000000,12913.500000,145802.000000,145793.500000,145792.000000,215.225000
max,247607.000000,2020-10-22 15:16:24,2020-10-22 18:06:54,2020-10-22 18:06:54,50.000000,99711.000000,995484.000000,995484.000000,145986.000000,145986.000000,145990.000000,16591.400000
std,7458.531611,NaN,NaN,NaN,3.897845,25706.521243,17127.636035,19965.233610,66.293879,61.921221,63.518170,332.753930


In [ ]:
cols_categoricas= ['type', 'status', 'resolution', 'sprint_id']
for col in cols_categoricas:
    print(f"--- Resumen de la columna: {col} ---")
    print(df_lsst[col].describe())
    print(df_lsst[col].value_counts())
    print("\n")

--- Resumen de la columna: type ---
count     10631
unique        6
top       Story
freq       8238
Name: type, dtype: object
type
Story             8238
Bug               1468
Improvement        575
Technical task     229
Epic               117
Milestone            4
Name: count, dtype: int64


--- Resumen de la columna: status ---
count     10631
unique        3
top        Done
freq      10005
Name: status, dtype: object
status
Done         10005
Won't Fix      325
Invalid        301
Name: count, dtype: int64


--- Resumen de la columna: resolution ---
count     10631
unique        1
top        Done
freq      10631
Name: resolution, dtype: object
resolution
Done    10631
Name: count, dtype: int64


--- Resumen de la columna: sprint_id ---
count            10631
unique             364
top       Not Assigned
freq              5910
Name: sprint_id, dtype: object
sprint_id
Not Assigned    5910
3159.0            55
2914.0            44
3153.0            43
2900.0            42
           

In [ ]:
df_lsst.head()

,issue_id,project_key,project_name,issue_key,type,status,resolution,creation_date,estimation_date,resolution_date,story_point,resolution_time_minutes,in_progress_minutes,total_effort_minutes,creator_id,reporter_id,assignee_id,sprint_id,text,total_effort_hours
46840,221116,DM,Lsstcorp Data management,DM-27270,Bug,Done,Done,2020-10-22 15:16:24,2020-10-22 16:28:46,2020-10-22 16:28:46,2.0,72,0,47,145715.0,145715.0,145716.0,Not Assigned,"""ap_verify failed on w_2020_43""\n\n""""""A datase...",0.78
46841,221117,DM,Lsstcorp Data management,DM-27269,Story,Done,Done,2020-10-22 14:51:07,2020-10-22 14:51:07,2020-10-22 17:13:45,0.2,142,0,62,145717.0,145717.0,145717.0,Not Assigned,"""Move controllers into Controllers directory""\...",1.03
46842,221121,DM,Lsstcorp Data management,DM-27265,Bug,Invalid,Done,2020-10-21 23:49:02,2020-10-22 00:14:33,2020-10-22 00:14:33,2.0,25,0,0,145722.0,145722.0,145718.0,Not Assigned,"""No TestID Control on the Connect Vi""\n\n""""""Th...",0.00
46843,221129,DM,Lsstcorp Data management,DM-27256,Story,Done,Done,2020-10-21 17:42:38,2020-10-21 17:42:38,2020-10-22 00:48:52,2.0,426,0,206,145720.0,145720.0,145720.0,Not Assigned,"""Add JSON support to butler Config""\n\n""""""Addi...",3.43
46844,221131,DM,Lsstcorp Data management,DM-27254,Story,Done,Done,2020-10-21 16:41:18,2020-10-21 19:50:37,2020-10-21 19:50:37,2.0,189,0,0,145732.0,145732.0,145731.0,Not Assigned,"""Get access to NSCA postgres RC2 gen3 database...",0.00


In [ ]:
cols_fechas = ['creation_date', 'estimation_date', 'resolution_date']
df[cols_fechas].describe()

,creation_date,estimation_date,resolution_date
count,133434,19307,133434
mean,2014-09-10 13:35:57.173674,2017-08-01 11:59:07.730149,2014-09-24 17:04:51.467129
min,2004-01-05 08:24:04,2009-03-09 13:17:19,2004-02-12 23:48:28
25%,2012-02-13 20:03:32,2016-02-19 17:56:43,2012-03-01 02:46:28.250000
50%,2015-03-23 17:14:01,2017-11-09 01:10:18,2015-04-06 20:48:46.500000
75%,2018-01-22 12:10:35.250000,2019-06-03 20:01:53.500000,2018-02-02 18:09:06
max,2020-10-22 15:16:24,2020-10-22 14:51:07,2020-10-22 22:22:28


In [ ]:
#Calculo de SPI y CPI
df_lsst["Año"] = df_lsst["estimation_date"].dt.year

# =========================================================================
# MAPEO DE MÉTRICAS EVM CON TUS COLUMNAS REALES
# =========================================================================

# PV (Planned Value): Todo lo estimado originalmente va aquí (en Story Points)
df_lsst["PV"] = df_lsst["story_point"]

# EV (Earned Value): Solo sumamos los puntos si el estatus es "Done" o terminado
# Nota: Si en tu Jira el estado final se llama 'Closed' o 'Resolved', cámbialo aquí
df_lsst["EV"] = df_lsst.apply(lambda row: row["story_point"] if str(row["status"]).strip().lower() == "done" else 0, axis=1)

# =========================================================================
# VALIDAR TASA HISTÓRICA
# =========================================================================
# Buscamos tareas que estén terminadas Y que tengan horas registradas > 0
tareas_con_tiempo = df_lsst[(df_lsst["status"] == "done") & (df_lsst["total_effort_hours"] > 0)]

horas_totales_done = tareas_con_tiempo["total_effort_hours"].sum()
sp_totales_done = tareas_con_tiempo["story_point"].sum()

# Si no hay horas registradas en todo el proyecto, evitamos la división entre cero
if sp_totales_done > 0 and horas_totales_done > 0:
    horas_por_sp_historico = horas_totales_done / sp_totales_done
else:
    # Si no hay datos de horas, usamos una estimación estándar (ej. 4 horas por punto)
    horas_por_sp_historico = 4.0

# AC (Actual Cost): Si las horas son cero, usamos los story_points como fallback
# para que no de infinito, asumiendo que costó lo que se planeó.
df_lsst["AC"] = df_lsst.apply(
    lambda row: row["total_effort_hours"] / horas_por_sp_historico if row["total_effort_hours"] > 0 else row["story_point"],
    axis=1
)

# =========================================================================
# AGRUPACIÓN Y CONTROL DE DIVISION POR 0
# =========================================================================
df_lsst_anual = df_lsst.groupby("Año")[["PV", "EV", "AC", "total_effort_hours"]].sum().reset_index()

df_lsst_anual["PV_Acumulado"] = df_lsst_anual["PV"].cumsum()
df_lsst_anual["EV_Acumulado"] = df_lsst_anual["EV"].cumsum()
df_lsst_anual["AC_Acumulado"] = df_lsst_anual["AC"].cumsum()
df_lsst_anual["Horas_Reales_Acumuladas"] = df_lsst_anual["total_effort_hours"].cumsum()

df_lsst_anual["SPI_Acumulado"] = (df_lsst_anual["EV_Acumulado"] / df_lsst_anual["PV_Acumulado"]).round(3)

# Calculamos el CPI y reemplazamos cualquier división entre cero (inf) por 1.0 o NaN
df_lsst_anual["CPI_Acumulado"] = (df_lsst_anual["EV_Acumulado"] / df_lsst_anual["AC_Acumulado"]).round(3)

# Esta línea mágica limpia los 'inf' si es que volviera a aparecer un cero remanente
import numpy as np
df_lsst_anual["CPI_Acumulado"] = df_lsst_anual["CPI_Acumulado"].replace([np.inf, -np.inf], np.nan)


# =========================================================================
# DESPLIEGUE DEL REPORTE
# =========================================================================
print(f"--- FACTOR DE RENDIMIENTO DE LSSTCORP ---")
print(f"En promedio, al equipo le toma {horas_por_sp_historico:.2f} horas reales completar 1 Story Point.\n")

print("=== REPORTE ACUMULADO HISTÓRICO DE PERFORMANCE ===")
columnas_finales = [
    "Año",
    "Horas_Reales_Acumuladas",
    "PV_Acumulado",
    "AC_Acumulado",
    "EV_Acumulado",
    "SPI_Acumulado",
    "CPI_Acumulado"
]
print(df_lsst_anual[columnas_finales].to_string(index=False))


--- FACTOR DE RENDIMIENTO DE LSSTCORP ---
En promedio, al equipo le toma 4.00 horas reales completar 1 Story Point.

=== REPORTE ACUMULADO HISTÓRICO DE PERFORMANCE ===
 Año  Horas_Reales_Acumuladas  PV_Acumulado  AC_Acumulado  EV_Acumulado  SPI_Acumulado  CPI_Acumulado
2014                 91599.29   1176.150000    23227.8225   1154.150000          0.981          0.050
2015                213238.79   4131.625000    54991.1725   3924.625000          0.950          0.071
2016                493226.34   9318.065997   126399.3100   8830.915997          0.948          0.070
2017                838585.45  15501.272997   214108.7135  14708.022997          0.949          0.069
2018               1139596.71  20881.752997   290765.9035  19901.502997          0.953          0.068
2019               1569495.30  27216.397997   399625.2260  25898.647997          0.952          0.065
2020               1878794.26  32042.523097   477679.7911  30420.773097          0.949          0.064


In [ ]:
#Revisión de fechas de milestones
# Filtrar solo los Milestones y seleccionar las columnas deseadas
milestones_df_lsst = df_lsst.loc[df_lsst['type'] == 'Milestone', ['type', 'estimation_date', 'creation_date', 'resolution_date']]

# Mostrar en Jupyter (usamos head() para ver las primeras o simplemente el nombre del df_lsst)
milestones_df_lsst.head(10)

,type,estimation_date,creation_date,resolution_date
49034,Milestone,2019-11-27 17:48:11,2019-11-27 16:54:12,2019-11-27 17:48:11
49048,Milestone,2020-01-07 19:45:59,2019-11-25 22:57:15,2020-01-07 19:45:59
50286,Milestone,2019-06-17 20:40:26,2019-04-22 17:44:51,2019-06-17 20:40:26
50300,Milestone,2019-06-17 19:13:29,2019-04-18 19:50:24,2019-06-17 19:13:29


In [ ]:
# guarda forzando la codificación utf-8 para que Tableau no se confunda
df_lsst.to_csv('lsst_ready_to_dashboard.csv', index=False, encoding='utf-8')

In [ ]:
df_lsst.to_excel('lsst_ready_to_dashboard.xlsx', index=False)